In [ ]:
# Midterm Project: EMNIST Letters Image Classification

Project steps:
- Process EMNIST Letters dataset and create a balanced subset of 2600 samples (100 per class)
- Shuffle data, split into 80% train and 20% test
- Extract HOG features with custom parameters
- Train SVM classifier and tune hyperparameters with Grid Search
- Evaluate performance on training and test sets with accuracy, precision, recall, and F1-score


In [1]:
# Blok 1: Membuat dataset seimbang (2600 sampel)
import pandas as pd

# Sesuaikan dengan path file datasetmu
input_csv = "emnist-letters-train.csv" 
output_csv = "new-emnist-letters-train.csv"

try:
    print("Membaca dataset asli...")
    # EMNIST Kaggle asli tidak ada header, kolom 0 adalah label (1-26)
    df = pd.read_csv(input_csv, header=None) 
    
    sampled_df = pd.DataFrame()
    
    # Looping untuk kelas 1 sampai 26 (A-Z)
    for label in range(1, 27):
        class_data = df[df[0] == label]
        # Ambil 100 sampel acak per kelas
        sampled_class_data = class_data.sample(n=100, random_state=42)  
        sampled_df = pd.concat([sampled_df, sampled_class_data])
    
    # Simpan ke CSV baru
    sampled_df.to_csv(output_csv, index=False, header=False)
    print(f"Sukses! Data seimbang disimpan di: {output_csv}")

except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Membaca dataset asli...
Sukses! Data seimbang disimpan di: new-emnist-letters-train.csv


In [2]:
# Blok 2: Load Data, Shuffle, dan Split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split, GridSearchCV, LeaveOneOut, cross_val_predict
from sklearn.svm import SVC
from skimage.feature import hog
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

# Load dataset baru
data_train = pd.read_csv(output_csv, header=None)
data_train = shuffle(data_train, random_state=42)

# Pisahkan fitur (piksel) dan label
X = data_train.iloc[:, 1:].values
y = data_train.iloc[:, 0].values

# Split 80% Train, 20% Test. Gunakan stratify agar proporsi kelas tetap rata.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dimensi X_train: {X_train.shape}")
print(f"Dimensi X_test: {X_test.shape}")

Dimensi X_train: (2080, 784)
Dimensi X_test: (520, 784)


In [3]:
def extract_hog_features(X_data):
    X_hog = np.zeros((X_data.shape[0], 1296)) 
    
    for i in range(X_data.shape[0]):
        img = X_data[i].reshape(28, 28).T 
        
        feature = hog(img, orientations=9, pixels_per_cell=(4, 4), 
                      cells_per_block=(2, 2), block_norm='L2')
        X_hog[i] = feature
        
    return X_hog

X_train_hog = extract_hog_features(X_train)

In [4]:
X_test_hog = extract_hog_features(X_test)

In [7]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.1, 1],
    'kernel': ['linear', 'rbf']
}

grid_search = GridSearchCV(estimator=SVC(random_state=42), param_grid=param_grid, 
                           cv=5, scoring='accuracy', verbose=2, n_jobs=-1)

grid_search.fit(X_train_hog, y_train)

best_svm_model = grid_search.best_estimator_

print(grid_search.best_params_)
print(grid_search.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
0.8399038461538462


In [8]:
loo = LeaveOneOut()
y_train_pred_loo = cross_val_predict(best_svm_model, X_train_hog, y_train, cv=loo, n_jobs=-1)

print(accuracy_score(y_train, y_train_pred_loo))
print(precision_score(y_train, y_train_pred_loo, average='weighted'))
print(recall_score(y_train, y_train_pred_loo, average='weighted'))
print(f1_score(y_train, y_train_pred_loo, average='weighted'))

KeyboardInterrupt: 

In [ ]:
y_test_pred = best_svm_model.predict(X_test_hog)

print(accuracy_score(y_test, y_test_pred))
print(precision_score(y_test, y_test_pred, average='weighted'))
print(recall_score(y_test, y_test_pred, average='weighted'))
print(f1_score(y_test, y_test_pred, average='weighted'))

In [ ]:
conf_mat = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(10, 8))

sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[chr(i+64) for i in range(1, 27)], 
            yticklabels=[chr(i+64) for i in range(1, 27)])
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(10):
    plt.subplot(2, 10, i + 1)
    img_asli = X_test[i].reshape(28, 28).T 
    plt.imshow(img_asli, cmap='gray')
    plt.axis('off')
    
    plt.subplot(2, 10, i + 11)
    _, hog_image = hog(img_asli, orientations=9, pixels_per_cell=(4, 4), 
                       cells_per_block=(2, 2), visualize=True, block_norm='L2')
    plt.imshow(hog_image, cmap='hot') 
    plt.axis('off')

plt.tight_layout()
plt.show()